# 01 — Data Extraction & Initial Profiling

**Project**: Hospital Readmission Rate Analysis (Diabetes 130-US Hospitals)  
**Team**: SEC-A, Group 3  
**Sector**: Healthcare  

---

## Objective

This notebook performs the **Extract** phase of the ETL pipeline:

1. Load the raw dataset from `data/raw/RAW_HEALTHCARE.csv`
2. Validate the schema — column names, data types, row count
3. Profile missing values, duplicates, and data quality indicators
4. Document the data source, provenance, and known limitations
5. Generate an extraction audit summary for downstream notebooks

> **Rule**: The raw file in `data/raw/` is **never modified**. All transformations occur in subsequent notebooks.

---

## Data Source

| Attribute | Detail |
|---|---|
| **Dataset** | Diabetes 130-US Hospitals for Years 1999–2008 |
| **Source** | UCI Machine Learning Repository / Kaggle |
| **Domain** | Healthcare — Inpatient diabetic encounters |
| **Records** | ~101,766 hospital encounters |
| **Features** | 50 columns (demographics, clinical, medications, diagnoses) |
| **Target** | `readmitted` — Whether a patient was readmitted (<30 days, >30 days, or No) |
| **Time Period** | 1999–2008 across 130 US hospitals |

## 1. Environment Setup

In [1]:
# ============================================================
# 01_extraction.ipynb — Environment Setup
# ============================================================
!pip install pandas 
!pip install numpy 
import pandas as pd
import numpy as np
import os
import warnings
from datetime import datetime

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 55)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 50)

print(f"Extraction started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version:  {np.__version__}")

zsh:1: command not found: pip


zsh:1: command not found: pip


Extraction started at: 2026-04-21 13:50:21
Pandas version: 2.3.3
NumPy version:  2.0.2


## 2. Load Raw Dataset

In [2]:
# ============================================================
# Define paths relative to project root
# ============================================================

RAW_DATA_PATH = os.path.join('..', 'data', 'raw', 'RAW_HEALTHCARE.csv')

# Verify file exists before loading
assert os.path.exists(RAW_DATA_PATH), f"Raw data file not found at: {RAW_DATA_PATH}"

# Load with '?' treated as a string (we handle it in cleaning)
df_raw = pd.read_csv(RAW_DATA_PATH, na_values=[], keep_default_na=False)

file_size_mb = os.path.getsize(RAW_DATA_PATH) / (1024 * 1024)
print(f"✅ Raw dataset loaded successfully")
print(f"   File size:  {file_size_mb:.2f} MB")
print(f"   Shape:      {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")

✅ Raw dataset loaded successfully


   File size:  18.27 MB
   Shape:      101,766 rows × 50 columns


## 3. Schema Validation

In [3]:
# ============================================================
# 3a. Expected schema (from data dictionary)
# ============================================================

EXPECTED_COLUMNS = [
    'encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight',
    'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
    'time_in_hospital', 'payer_code', 'medical_specialty',
    'num_lab_procedures', 'num_procedures', 'num_medications',
    'number_outpatient', 'number_emergency', 'number_inpatient',
    'diag_1', 'diag_2', 'diag_3', 'number_diagnoses',
    'max_glu_serum', 'A1Cresult',
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
    'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
    'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton',
    'insulin', 'glyburide-metformin', 'glipizide-metformin',
    'glimepiride-pioglitazone', 'metformin-rosiglitazone',
    'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted'
]

# Validate column presence
actual_cols = list(df_raw.columns)
missing_cols = set(EXPECTED_COLUMNS) - set(actual_cols)
extra_cols = set(actual_cols) - set(EXPECTED_COLUMNS)

print(f"Expected columns: {len(EXPECTED_COLUMNS)}")
print(f"Actual columns:   {len(actual_cols)}")
print(f"Missing columns:  {missing_cols if missing_cols else 'None ✅'}")
print(f"Extra columns:    {extra_cols if extra_cols else 'None ✅'}")

Expected columns: 50
Actual columns:   50
Missing columns:  None ✅
Extra columns:    None ✅


In [4]:
# ============================================================
# 3b. Column data types
# ============================================================

schema_df = pd.DataFrame({
    'column': df_raw.columns,
    'dtype': df_raw.dtypes.values,
    'non_null_count': df_raw.count().values,
    'null_count': df_raw.isnull().sum().values,
    'unique_count': df_raw.nunique().values,
    'sample_value': [df_raw[col].iloc[0] for col in df_raw.columns]
})

schema_df

,column,dtype,non_null_count,null_count,unique_count,sample_value
0,encounter_id,int64,101766,0,101766,2278392
1,patient_nbr,int64,101766,0,71518,8222157
2,race,object,101766,0,6,Caucasian
3,gender,object,101766,0,3,Female
4,age,object,101766,0,10,[0-10)
5,weight,object,101766,0,10,?
6,admission_type_id,int64,101766,0,8,6
7,discharge_disposition_id,int64,101766,0,26,25
8,admission_source_id,int64,101766,0,17,1
9,time_in_hospital,int64,101766,0,14,1


## 4. Initial Data Preview

In [5]:
# ============================================================
# 4a. First 5 rows — verify structure
# ============================================================

df_raw.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,?,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,?,?,1,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,?,?,59,0,18,0,0,0,276,250.01,255,9,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,?,?,11,5,13,2,0,1,648,250,V27,6,None,None,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,?,?,44,1,16,0,0,0,8,250.43,403,7,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,?,?,51,0,8,0,0,0,197,157,250,5,None,None,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [6]:
# ============================================================
# 4b. Last 5 rows — check for truncation or trailing issues
# ============================================================

df_raw.tail()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
101761,443847548,100162476,AfricanAmerican,Male,[70-80),?,1,3,7,3,MC,?,51,0,16,0,0,0,250.13,291,458,9,None,>8,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Down,No,No,No,No,No,Ch,Yes,>30
101762,443847782,74694222,AfricanAmerican,Female,[80-90),?,1,4,5,5,MC,?,33,3,18,0,0,1,560,276,787,9,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Yes,NO
101763,443854148,41088789,Caucasian,Male,[70-80),?,1,1,7,1,MC,?,53,0,9,1,0,0,38,590,296,13,None,None,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Down,No,No,No,No,No,Ch,Yes,NO
101764,443857166,31693671,Caucasian,Female,[80-90),?,2,3,7,10,MC,Surgery-General,45,2,21,0,0,1,996,285,998,9,None,None,No,No,No,No,No,No,Steady,No,No,Steady,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
101765,443867222,175429310,Caucasian,Male,[70-80),?,1,1,7,6,?,?,13,3,3,0,0,0,530,530,787,9,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO


In [7]:
# ============================================================
# 4c. Random sample — check data variety
# ============================================================

df_raw.sample(5, random_state=42)

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
35956,110939484,19274094,Caucasian,Female,[70-80),?,1,1,6,11,UN,InternalMedicine,68,0,20,0,0,0,250.8,599,263,5,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Yes,NO
60927,170328306,65634327,Caucasian,Male,[50-60),?,1,1,1,1,HM,?,20,0,7,0,0,0,780,427,E941,8,None,None,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
79920,245688426,100657359,Caucasian,Female,[60-70),?,3,6,1,4,HM,?,21,3,23,1,0,2,715,733,724,7,None,None,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
50078,150826224,83144448,Caucasian,Male,[30-40),?,2,1,1,12,CP,Gastroenterology,28,0,19,0,0,1,494,277,117,7,None,None,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,>30
44080,135993852,65234214,AfricanAmerican,Female,[60-70),?,1,2,7,1,?,?,21,0,6,0,0,0,403,584,428,7,None,None,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,<30


## 5. Descriptive Statistics

In [8]:
# ============================================================
# 5a. Numerical columns — summary statistics
# ============================================================

numerical_cols = df_raw.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numerical columns ({len(numerical_cols)}): {numerical_cols}\n")

df_raw[numerical_cols].describe().round(2)

Numerical columns (13): ['encounter_id', 'patient_nbr', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']



,encounter_id,patient_nbr,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses
count,1.017660e+05,1.017660e+05,101766.00,101766.00,101766.00,101766.00,101766.00,101766.00,101766.00,101766.00,101766.00,101766.00,101766.00
mean,1.652016e+08,5.433040e+07,2.02,3.72,5.75,4.40,43.10,1.34,16.02,0.37,0.20,0.64,7.42
std,1.026403e+08,3.869636e+07,1.45,5.28,4.06,2.99,19.67,1.71,8.13,1.27,0.93,1.26,1.93
min,1.252200e+04,1.350000e+02,1.00,1.00,1.00,1.00,1.00,0.00,1.00,0.00,0.00,0.00,1.00
25%,8.496119e+07,2.341322e+07,1.00,1.00,1.00,2.00,31.00,0.00,10.00,0.00,0.00,0.00,6.00
50%,1.523890e+08,4.550514e+07,1.00,1.00,7.00,4.00,44.00,1.00,15.00,0.00,0.00,0.00,8.00
75%,2.302709e+08,8.754595e+07,3.00,4.00,7.00,6.00,57.00,2.00,20.00,0.00,0.00,1.00,9.00
max,4.438672e+08,1.895026e+08,8.00,28.00,25.00,14.00,132.00,6.00,81.00,42.00,76.00,21.00,16.00


In [9]:
# ============================================================
# 5b. Categorical columns — value counts overview
# ============================================================

categorical_cols = df_raw.select_dtypes(include=['object']).columns.tolist()
print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}\n")

for col in categorical_cols:
    n_unique = df_raw[col].nunique()
    top_val = df_raw[col].value_counts().index[0]
    top_pct = (df_raw[col].value_counts().iloc[0] / len(df_raw) * 100)
    print(f"  {col:35s} | {n_unique:4d} unique | Top: '{top_val}' ({top_pct:.1f}%)")

Categorical columns (37): ['race', 'gender', 'age', 'weight', 'payer_code', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted']

  race                                |    6 unique | Top: 'Caucasian' (74.8%)
  gender                              |    3 unique | Top: 'Female' (53.8%)
  age                                 |   10 unique | Top: '[70-80)' (25.6%)
  weight                              |   10 unique | Top: '?' (96.9%)
  payer_code                          |   18 unique | Top: '?' (39.6%)
  medical_specialty                   |   73

  acarbose                            |    4 unique | Top: 'No' (99.7%)
  miglitol                            |    4 unique | Top: 'No' (100.0%)
  troglitazone                        |    2 unique | Top: 'No' (100.0%)
  tolazamide                          |    3 unique | Top: 'No' (100.0%)
  examide                             |    1 unique | Top: 'No' (100.0%)
  citoglipton                         |    1 unique | Top: 'No' (100.0%)
  insulin                             |    4 unique | Top: 'No' (46.6%)


  glyburide-metformin                 |    4 unique | Top: 'No' (99.3%)
  glipizide-metformin                 |    2 unique | Top: 'No' (100.0%)
  glimepiride-pioglitazone            |    2 unique | Top: 'No' (100.0%)
  metformin-rosiglitazone             |    2 unique | Top: 'No' (100.0%)
  metformin-pioglitazone              |    2 unique | Top: 'No' (100.0%)
  change                              |    2 unique | Top: 'No' (53.8%)
  diabetesMed                         |    2 unique | Top: 'Yes' (77.0%)
  readmitted                          |    3 unique | Top: 'NO' (53.9%)


## 6. Missing Value & Placeholder Analysis

This dataset uses `?` as a placeholder for missing values instead of standard NaN. We document this here and handle conversion in `02_cleaning.ipynb`.

In [10]:
# ============================================================
# 6a. Count '?' placeholders across all columns
# ============================================================

placeholder_counts = {}
for col in df_raw.columns:
    if df_raw[col].dtype == 'object':
        q_count = (df_raw[col] == '?').sum()
        if q_count > 0:
            placeholder_counts[col] = q_count

missing_df = pd.DataFrame({
    'column': placeholder_counts.keys(),
    'missing_count': placeholder_counts.values(),
    'missing_pct': [v / len(df_raw) * 100 for v in placeholder_counts.values()]
}).sort_values('missing_pct', ascending=False).reset_index(drop=True)

missing_df['missing_pct'] = missing_df['missing_pct'].round(2)
print("Columns with '?' placeholder values:\n")
missing_df

Columns with '?' placeholder values:



,column,missing_count,missing_pct
0,weight,98569,96.86
1,medical_specialty,49949,49.08
2,payer_code,40256,39.56
3,race,2273,2.23
4,diag_3,1423,1.40
5,diag_2,358,0.35
6,diag_1,21,0.02


In [11]:
# ============================================================
# 6b. 'None' as a categorical value (lab results)
# max_glu_serum and A1Cresult use 'None' to indicate 'not measured'
# ============================================================

print("max_glu_serum value counts:")
print(df_raw['max_glu_serum'].value_counts())
print(f"\nA1Cresult value counts:")
print(df_raw['A1Cresult'].value_counts())

max_glu_serum value counts:
max_glu_serum
None    96420
Norm     2597
>200     1485
>300     1264
Name: count, dtype: int64

A1Cresult value counts:
A1Cresult
None    84748
>8       8216
Norm     4990
>7       3812
Name: count, dtype: int64


## 7. Duplicate Analysis

In [12]:
# ============================================================
# 7a. Exact row duplicates
# ============================================================

n_exact_dupes = df_raw.duplicated().sum()
print(f"Exact duplicate rows: {n_exact_dupes}")

# ============================================================
# 7b. Duplicate encounter_ids (should be unique per encounter)
# ============================================================

n_encounter_dupes = df_raw['encounter_id'].duplicated().sum()
print(f"Duplicate encounter_id values: {n_encounter_dupes}")

# ============================================================
# 7c. Patients with multiple encounters
# ============================================================

patient_encounter_counts = df_raw['patient_nbr'].value_counts()
multi_encounter = patient_encounter_counts[patient_encounter_counts > 1]

print(f"\nTotal unique patients:             {df_raw['patient_nbr'].nunique():,}")
print(f"Patients with multiple encounters:  {len(multi_encounter):,}")
print(f"Max encounters per patient:         {patient_encounter_counts.max()}")
print(f"\nEncounter distribution per patient:")
print(patient_encounter_counts.value_counts().sort_index().head(10))

Exact duplicate rows: 0
Duplicate encounter_id values: 0

Total unique patients:             71,518
Patients with multiple encounters:  16,773
Max encounters per patient:         40

Encounter distribution per patient:
count
1     54745
2     10434
3      3328
4      1421
5       717
6       346
7       207
8       111
9        70
10       42
Name: count, dtype: int64


## 8. Target Variable Distribution

In [13]:
# ============================================================
# 8. Readmission target — class balance check
# ============================================================

target_dist = df_raw['readmitted'].value_counts()
target_pct = df_raw['readmitted'].value_counts(normalize=True).mul(100).round(2)

target_summary = pd.DataFrame({
    'count': target_dist,
    'percentage': target_pct
})

print("Target Variable Distribution (readmitted):\n")
print(target_summary)
print(f"\n--- Business Context ---")
print(f"30-day readmission rate: {target_pct.get('<30', 0):.2f}%")
print(f"This is the critical metric — CMS penalises hospitals with high 30-day readmission rates.")

Target Variable Distribution (readmitted):

            count  percentage
readmitted                   
NO          54864       53.91
>30         35545       34.93
<30         11357       11.16

--- Business Context ---
30-day readmission rate: 11.16%
This is the critical metric — CMS penalises hospitals with high 30-day readmission rates.


## 9. Column Classification

Categorising columns by their analytical role to guide downstream cleaning and analysis.

In [14]:
# ============================================================
# 9. Logical column grouping
# ============================================================

column_groups = {
    'identifiers': ['encounter_id', 'patient_nbr'],
    
    'demographics': ['race', 'gender', 'age', 'weight'],
    
    'admission_info': [
        'admission_type_id', 'discharge_disposition_id',
        'admission_source_id', 'time_in_hospital', 'payer_code'
    ],
    
    'clinical_utilisation': [
        'medical_specialty', 'num_lab_procedures', 'num_procedures',
        'num_medications', 'number_outpatient', 'number_emergency',
        'number_inpatient', 'number_diagnoses'
    ],
    
    'diagnoses': ['diag_1', 'diag_2', 'diag_3'],
    
    'lab_results': ['max_glu_serum', 'A1Cresult'],
    
    'medications': [
        'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
        'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
        'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
        'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton',
        'insulin', 'glyburide-metformin', 'glipizide-metformin',
        'glimepiride-pioglitazone', 'metformin-rosiglitazone',
        'metformin-pioglitazone'
    ],
    
    'treatment_flags': ['change', 'diabetesMed'],
    
    'target': ['readmitted']
}

for group, cols in column_groups.items():
    print(f"\n{group.upper()} ({len(cols)} cols): {cols}")


IDENTIFIERS (2 cols): ['encounter_id', 'patient_nbr']

DEMOGRAPHICS (4 cols): ['race', 'gender', 'age', 'weight']

ADMISSION_INFO (5 cols): ['admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'payer_code']

CLINICAL_UTILISATION (8 cols): ['medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']

DIAGNOSES (3 cols): ['diag_1', 'diag_2', 'diag_3']

LAB_RESULTS (2 cols): ['max_glu_serum', 'A1Cresult']

MEDICATIONS (23 cols): ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone']

TREATMENT_FLAGS (2 cols): ['change', 'dia

## 10. Medication Columns — Variance Check

Several medication columns may have near-zero variance (almost all `No`), making them uninformative. We flag them here for potential removal in `02_cleaning.ipynb`.

In [15]:
# ============================================================
# 10. Medication columns — check for near-zero variance
# ============================================================

med_cols = column_groups['medications']

med_summary = []
for col in med_cols:
    vc = df_raw[col].value_counts(normalize=True)
    no_pct = vc.get('No', 0) * 100
    med_summary.append({
        'medication': col,
        'pct_No': round(no_pct, 2),
        'n_unique': df_raw[col].nunique(),
        'dominant_value': vc.index[0]
    })

med_df = pd.DataFrame(med_summary).sort_values('pct_No', ascending=False)

print("Medication columns — 'No' prevalence:\n")
print(med_df.to_string(index=False))

# Flag columns with >99% 'No' — candidates for removal
low_variance_meds = med_df[med_df['pct_No'] > 99]['medication'].tolist()
print(f"\n⚠️ Near-zero variance medications (>99% 'No'): {low_variance_meds}")
print(f"   These {len(low_variance_meds)} columns add negligible analytical value and will be reviewed for removal.")

Medication columns — 'No' prevalence:

              medication  pct_No  n_unique dominant_value
  metformin-pioglitazone  100.00         2             No
 metformin-rosiglitazone  100.00         2             No
glimepiride-pioglitazone  100.00         2             No
           acetohexamide  100.00         2             No
             citoglipton  100.00         1             No
                 examide  100.00         1             No
            troglitazone  100.00         2             No
     glipizide-metformin   99.99         2             No
             tolbutamide   99.98         2             No
                miglitol   99.96         4             No
              tolazamide   99.96         3             No
          chlorpropamide   99.92         4             No
                acarbose   99.70         4             No
     glyburide-metformin   99.31         4             No
             nateglinide   99.31         4             No
             repaglinide   98.49 

## 11. Extraction Audit Summary

In [16]:
# ============================================================
# 11. Extraction audit log
# ============================================================

audit = {
    'extraction_timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'source_file': RAW_DATA_PATH,
    'file_size_mb': round(file_size_mb, 2),
    'total_rows': df_raw.shape[0],
    'total_columns': df_raw.shape[1],
    'numerical_columns': len(numerical_cols),
    'categorical_columns': len(categorical_cols),
    'exact_duplicate_rows': int(n_exact_dupes),
    'duplicate_encounter_ids': int(n_encounter_dupes),
    'unique_patients': int(df_raw['patient_nbr'].nunique()),
    'columns_with_placeholders': len(placeholder_counts),
    'target_classes': dict(target_dist),
    'low_variance_medications': low_variance_meds
}

print("=" * 60)
print("         EXTRACTION AUDIT SUMMARY")
print("=" * 60)
for key, val in audit.items():
    print(f"  {key:35s}: {val}")
print("=" * 60)

         EXTRACTION AUDIT SUMMARY
  extraction_timestamp               : 2026-04-21 13:50:22
  source_file                        : ../data/raw/RAW_HEALTHCARE.csv
  file_size_mb                       : 18.27
  total_rows                         : 101766
  total_columns                      : 50
  numerical_columns                  : 13
  categorical_columns                : 37
  exact_duplicate_rows               : 0
  duplicate_encounter_ids            : 0
  unique_patients                    : 71518
  columns_with_placeholders          : 7
  target_classes                     : {'NO': np.int64(54864), '>30': np.int64(35545), '<30': np.int64(11357)}
  low_variance_medications           : ['metformin-pioglitazone', 'metformin-rosiglitazone', 'glimepiride-pioglitazone', 'acetohexamide', 'citoglipton', 'examide', 'troglitazone', 'glipizide-metformin', 'tolbutamide', 'miglitol', 'tolazamide', 'chlorpropamide', 'acarbose', 'glyburide-metformin', 'nateglinide']


## 12. Key Observations from Extraction

| # | Observation | Implication for Cleaning |
|---|---|---|
| 1 | `weight` is missing for **96.9%** of records | **Drop column** — insufficient data for any analysis |
| 2 | `payer_code` is missing for **39.6%** of records | **Drop column** — payer info is not central to readmission analysis |
| 3 | `medical_specialty` is missing for **49.1%** | **Recode** to `Unknown` — specialty still carries analytical value |
| 4 | `max_glu_serum` (94.8%) and `A1Cresult` (83.3%) show `None` | **Recode `None` → `Not Measured`** — this is a valid clinical state |
| 5 | `?` is used as a missing value placeholder | **Replace all `?` with NaN** in cleaning |
| 6 | `gender` has 3 `Unknown/Invalid` entries | **Drop these 3 rows** — negligible loss |
| 7 | Multiple encounters per patient exist | **Keep only the first encounter** per patient to prevent data leakage |
| 8 | Several medication columns have >99% `No` values | **Drop near-zero variance columns** — no analytical signal |
| 9 | `admission_type_id`, `discharge_disposition_id`, `admission_source_id` are coded as integers | **Map to meaningful labels** using ICD reference tables |
| 10 | Target is imbalanced: `<30` is only ~11% | **Note for analysis** — weight metrics appropriately |

---

**Next Step** → `02_cleaning.ipynb` — Execute the full cleaning pipeline based on these observations.